In [1]:
%pip uninstall -y bitsandbytes torchvision torchaudio trl
%pip install -U "transformers>=4.45,<5" "peft>=0.12,<1" "accelerate>=0.34,<2" "datasets>=2.20,<4" "sentencepiece" "safetensors"


Note: you may need to restart the kernel to use updated packages.
  Using cached fsspec-2025.3.0-py3-none-any.whl.metadata (11 kB)
Using cached fsspec-2025.3.0-py3-none-any.whl (193 kB)
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2026.4.0
    Uninstalling fsspec-2026.4.0:
      Successfully uninstalled fsspec-2026.4.0

[notice] A new release of pip is available: 26.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model

print("CUDA:", torch.cuda.is_available())
print("Imports OK")


/Users/alexanderchin/code/LLM_Personalization---ACM-AI-Team4/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA: False
Imports OK


In [3]:
%pip uninstall -y torchao


Note: you may need to restart the kernel to use updated packages.


In [4]:
"""
Notebook-friendly LoRA training script for Colab/Jupyter on a T4 GPU.

Expected dataset schema:
- jsonl: one JSON object per line with "prompt" and "response"
- csv: columns named "prompt" and "response"

Suggested Colab install cell:
%pip install -U "transformers>=4.45,<5" "peft>=0.12,<1" "accelerate>=0.34,<2" "datasets>=2.20,<4" "sentencepiece" "safetensors"

Then restart the runtime before running this file.
"""

import csv
import importlib.metadata
import json
import os
import shutil
from pathlib import Path

try:
    import torch
except ImportError as exc:
    raise ImportError(
        "PyTorch is not installed. Run the Colab install cell, then restart the runtime."
    ) from exc

try:
    from datasets import Dataset
except ImportError as exc:
    raise ImportError(
        "The 'datasets' package is missing. Run the Colab install cell, then restart the runtime."
    ) from exc

try:
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
except ImportError as exc:
    raise ImportError(
        "PEFT import failed. This is usually a PEFT/Transformers version mismatch. "
        "Reinstall the pinned packages and restart the runtime."
    ) from exc

try:
    from transformers import (
        AutoModelForCausalLM,
        AutoTokenizer,
        DataCollatorForLanguageModeling,
        Trainer,
        TrainingArguments,
    )
except ImportError as exc:
    raise ImportError(
        "Transformers imports failed. Reinstall the pinned packages and restart the runtime."
    ) from exc


# %%
# ---------------------------------------------------------------------------
# Notebook config
# ---------------------------------------------------------------------------
BASE_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
DATASET_PATH = "trump_data/trump_tweets_with_prompts.json"
OUTPUT_ADAPTER_DIR = "./lora_Trump_v1"
OPTIONAL_DRIVE_OUTPUT_DIR = None

ALLOW_FALLBACK_SAMPLE = True
MAX_TRAIN_SAMPLES = None
USE_4BIT = False

MAX_LENGTH = 384
NUM_TRAIN_EPOCHS = 3
PER_DEVICE_TRAIN_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 2e-4
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

MAX_NEW_TOKENS = 120
TEMPERATURE = 0.4
TOP_P = 0.9

EVAL_PROMPTS = [
    "Compose a polite letter declining an invitation I cannot accept.",
    "Draft a short note thanking a colleague for a recent kindness.",
    "What do you think about the proper role of state legislatures in a republic?",
    "What's your opinion on cryptocurrency? ",
]

FALLBACK_EXAMPLES = [
    {"prompt": "Describe the sea at dawn.", "response": "The sea at dawn glows like spilled gold, and every wave looks fit to carry a bold rogue toward fortune."},
    {"prompt": "Write a short poem about canaries.", "response": "Bright canaries sing in the rigging high, like coins of song tossed into the sky."},
    {"prompt": "Give advice before a hard journey.", "response": "Pack courage, keep close to trusted mates, and mind the sky, for trouble often sails ahead of itself."},
    {"prompt": "Describe a hidden island.", "response": "The island lies behind black rock and quiet surf, green with secrets and silent as buried treasure."},
]


# %%
# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def print_dependency_versions():
    package_names = ["torch", "transformers", "peft", "accelerate", "datasets", "bitsandbytes"]
    print("Dependency versions:")
    for package_name in package_names:
        try:
            version = importlib.metadata.version(package_name)
        except importlib.metadata.PackageNotFoundError:
            version = "not installed"
        print(f"- {package_name}: {version}")


def require_cuda():
    if not torch.cuda.is_available():
        raise RuntimeError(
            "This notebook script is intended for GPU training. "
            "Switch Colab runtime to T4 GPU or run it on a CUDA machine."
        )


def require_bitsandbytes_if_needed():
    if not USE_4BIT:
        return

    try:
        import bitsandbytes  # noqa: F401
    except ImportError as exc:
        raise ImportError(
            "bitsandbytes is required only when USE_4BIT=True. "
            "Either install bitsandbytes cleanly or leave USE_4BIT=False."
        ) from exc


def load_rows_from_file(dataset_path: str):
    path = Path(dataset_path)
    if not path.exists():
        if ALLOW_FALLBACK_SAMPLE:
            print(f"Dataset file not found at {dataset_path}. Using tiny fallback sample for smoke testing.")
            return FALLBACK_EXAMPLES
        raise FileNotFoundError(
            f"Dataset file not found at {dataset_path}. "
            "Create a jsonl/csv file with 'prompt' and 'response' fields."
        )

    if path.suffix.lower() == ".json":
        rows = []
        with path.open("r", encoding="utf-8") as handle:
            data = json.load(handle)
            for line_number, row in enumerate(data, start=1):
                mapped_row = {
                    "prompt": row.get("prompt", ""),
                    "response": row.get("output", row.get("response", ""))
                }
                validate_row(mapped_row, source=f"{dataset_path}:{line_number}")
                rows.append({"prompt": mapped_row["prompt"], "response": mapped_row["response"]})
        return rows

    if path.suffix.lower() == ".jsonl":
        rows = []
        with path.open("r", encoding="utf-8") as handle:
            for line_number, line in enumerate(handle, start=1):
                if not line.strip():
                    continue
                try:
                    row = json.loads(line)
                except json.JSONDecodeError as exc:
                    raise ValueError(f"Invalid JSON on line {line_number}: {exc}") from exc
                validate_row(row, source=f"{dataset_path}:{line_number}")
                rows.append({"prompt": row["prompt"], "response": row["response"]})
        return rows

    if path.suffix.lower() == ".csv":
        rows = []
        with path.open("r", encoding="utf-8", newline="") as handle:
            reader = csv.DictReader(handle)
            for line_number, row in enumerate(reader, start=2):
                validate_row(row, source=f"{dataset_path}:{line_number}")
                rows.append({"prompt": row["prompt"], "response": row["response"]})
        return rows

    raise ValueError(
        f"Unsupported dataset format for {dataset_path}. "
        "Use .json, .jsonl or .csv with 'prompt' and 'response' (or 'output')."
    )


def validate_row(row, source: str):
    if "prompt" not in row or "response" not in row:
        raise ValueError(f"Missing 'prompt' or 'response' in {source}.")
    # Prompts can be empty depending on the dataset structure, but response should not be.
    if not isinstance(row["response"], str) or not row["response"].strip():
        raise ValueError(f"Empty response in {source}.")


def maybe_trim_rows(rows):
    if MAX_TRAIN_SAMPLES is None:
        return rows
    return rows[:MAX_TRAIN_SAMPLES]


def build_train_texts(rows, tokenizer):
    texts = []
    for row in rows:
        messages = [
            {"role": "user", "content": row["prompt"]},
            {"role": "assistant", "content": row["response"]},
        ]
        if hasattr(tokenizer, "apply_chat_template"):
            text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False,
            )
        else:
            text = f"User: {row['prompt']}\nAssistant: {row['response']}"
        texts.append(text)
    return texts


def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )


def generate_reply(model, tokenizer, prompt: str):
    messages = [{"role": "user", "content": prompt}]
    if hasattr(tokenizer, "apply_chat_template"):
        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
        )
    else:
        inputs = tokenizer(prompt, return_tensors="pt")

    inputs = {key: value.to(model.device) for key, value in inputs.items()}
    input_length = inputs["input_ids"].shape[1]

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = output[0][input_length:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()


def print_comparison(title: str, outputs):
    print(f"\n=== {title} ===")
    for prompt, text in outputs:
        print(f"\nPrompt: {prompt}")
        print(text)


def evaluate_prompts(model, tokenizer):
    outputs = []
    model.eval()
    for prompt in EVAL_PROMPTS:
        outputs.append((prompt, generate_reply(model, tokenizer, prompt)))
    return outputs


def save_outputs(model, tokenizer, output_dir: str):
    os.makedirs(output_dir, exist_ok=True)
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)

    if OPTIONAL_DRIVE_OUTPUT_DIR:
        shutil.copytree(output_dir, OPTIONAL_DRIVE_OUTPUT_DIR, dirs_exist_ok=True)
        print(f"Copied adapter to {OPTIONAL_DRIVE_OUTPUT_DIR}")


# %%
# ---------------------------------------------------------------------------
# GPU check and dataset load
# ---------------------------------------------------------------------------
print_dependency_versions()
require_cuda()
require_bitsandbytes_if_needed()
print(f"CUDA device: {torch.cuda.get_device_name(0)}")
print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / (1024 ** 3):.2f} GB")

rows = maybe_trim_rows(load_rows_from_file(DATASET_PATH))
print(f"Loaded {len(rows)} training examples.")


# %%
# ---------------------------------------------------------------------------
# Tokenizer and base model
# ---------------------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

if USE_4BIT:
    from transformers import BitsAndBytesConfig

    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )

    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        quantization_config=quant_config,
        device_map="auto",
    )
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto",
    )

base_outputs = evaluate_prompts(base_model, tokenizer)
print_comparison("Base Model Outputs", base_outputs)


# %%
# ---------------------------------------------------------------------------
# Dataset build
# ---------------------------------------------------------------------------
train_texts = build_train_texts(rows, tokenizer)
dataset = Dataset.from_dict({"text": train_texts})
dataset = dataset.map(tokenize_batch, batched=True, remove_columns=["text"])

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)


# %%
# ---------------------------------------------------------------------------
# LoRA setup
# ---------------------------------------------------------------------------
base_model.config.use_cache = False
if USE_4BIT:
    base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=True)
else:
    base_model.gradient_checkpointing_enable()
    base_model.enable_input_require_grads()

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()


# %%
# ---------------------------------------------------------------------------
# Trainer
# ---------------------------------------------------------------------------
training_args = TrainingArguments(
    output_dir=OUTPUT_ADAPTER_DIR,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    logging_steps=5,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    fp16=False,
    bf16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit" if USE_4BIT else "adamw_torch",
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    max_grad_norm=0.3,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=data_collator,
)


# %%
# ---------------------------------------------------------------------------
# Train
# ---------------------------------------------------------------------------
trainer.train()


# %%
# ---------------------------------------------------------------------------
# Evaluate tuned model and save adapter
# ---------------------------------------------------------------------------
model.config.use_cache = True
tuned_outputs = evaluate_prompts(model, tokenizer)
print_comparison("Tuned Model Outputs", tuned_outputs)

save_outputs(model, tokenizer, OUTPUT_ADAPTER_DIR)
print(f"\nAdapter saved to {OUTPUT_ADAPTER_DIR}")


Dependency versions:
- torch: 2.11.0
- transformers: 4.57.6
- peft: 0.19.1
- accelerate: 1.13.0
- datasets: 3.6.0
- bitsandbytes: not installed


RuntimeError: This notebook script is intended for GPU training. Switch Colab runtime to T4 GPU or run it on a CUDA machine.

In [5]:
# Interactive prompt cell for notebook use

def notebook_generate(model, tokenizer, prompt, max_new_tokens=120, temperature=0.4, top_p=0.9):
    messages = [{"role": "user", "content": prompt}]

    if hasattr(tokenizer, "apply_chat_template"):
        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
        )
    else:
        inputs = tokenizer(prompt, return_tensors="pt")

    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output[0][input_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


while True:
    prompt = input("Prompt> ").strip()
    if not prompt or prompt.lower() in {"exit", "quit"}:
        print("Stopped.")
        break

    print()
    print(notebook_generate(model, tokenizer, prompt))
    print()


NameError: name 'model' is not defined

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_PATH = "/content/lora_Jefferson_v1"

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

print("Loading base model:", BASE_MODEL_NAME)
print("Loading adapter:", ADAPTER_PATH)
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=dtype,
    device_map="auto" if device == "cuda" else None,
)

if device != "cuda":
    base_model.to(device)

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()


def notebook_generate(model, tokenizer, prompt, max_new_tokens=120, temperature=0.4, top_p=0.9):
    messages = [{"role": "user", "content": prompt}]

    if hasattr(tokenizer, "apply_chat_template"):
        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
        )
    else:
        inputs = tokenizer(prompt, return_tensors="pt")

    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output[0][input_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


while True:
    prompt = input("Prompt> ").strip()
    if not prompt or prompt.lower() in {"exit", "quit"}:
        print("Stopped.")
        break

    print()
    print(notebook_generate(model, tokenizer, prompt))
    print()


In [ ]:
from google.colab import drive
drive.mount('/content/drive')